## Assessed dissertation model runner


In [ ]:
# === One-cell runner for your DCM estimations (Colab-ready) ===
# - Auth + BigQuery fetch
# - Config-driven MNL estimation with user-cluster robust SE
# - Weighted LL/LL0, McFadden R^2, per-user CS and ΔCS under Sponsor=0
# - Excludes Model3 V3_1 (as requested); includes V3_2.

!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from numpy.linalg import inv

PROJECT_ID = "my-dissertation-470916"

# ---- Model registry (table, X columns, param names, zeroed terms in Sponsor=0 CF)
REGISTRY = {
    # Baseline
    "M1_baseline": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est",
        "x": ["listen_score_z","timecost_z","sponsor"],
        "names": ["ListenScore","TimeCost","Sponsor"],
        "cf_zero": ["Sponsor"]
    },
    # Model 2 variants
    "M2V1": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V2",
        "x": ["listen_score_z","timecost_z","sponsor","sponsor_highIncome","time_highFreq"],
        "names": ["ListenScore","TimeCost","Sponsor","Sponsor×HighIncome","TimeCost×HighFreq"],
        "cf_zero": ["Sponsor","Sponsor×HighIncome"]
    },
    # V3_2 (NOT V3_1)
    "M2V2": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V3",
        "x": ["listen_score_z","timecost_z","sponsor","listen_young","time_highIncome","listen_highFreq"],
        "names": ["ListenScore","TimeCost","Sponsor","ListenScore×Young","TimeCost×HighIncome","ListenScore×HighFreq"],
        "cf_zero": ["Sponsor"]  # V3_2’de sponsor interaction yok; sadece Sponsor kapanır
    },
    "M2V3": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V4",
        "x": ["listen_score_z","timecost_z","sponsor","listen_highEdu","time_young","sponsor_female"],
        "names": ["ListenScore","TimeCost","Sponsor","ListenScore×HighEdu","TimeCost×Young","Sponsor×Female"],
        "cf_zero": ["Sponsor","Sponsor×Female"]
    },
    "M2V4": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V5",
        "x": ["listen_score_z","timecost_z","sponsor","time_highEdu","listen_highIncome","sponsor_highFreq"],
        "names": ["ListenScore","TimeCost","Sponsor","TimeCost×HighEdu","ListenScore×HighIncome","Sponsor×HighFreq"],
        "cf_zero": ["Sponsor","Sponsor×HighFreq"]
    },
    "M2V5": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V6",
        "x": ["listen_score_z","timecost_z","sponsor","sponsor_highEdu","time_older"],
        "names": ["ListenScore","TimeCost","Sponsor","Sponsor×HighEdu","TimeCost×Older"],
        "cf_zero": ["Sponsor","Sponsor×HighEdu"]
    },
    "M2V6": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V7",
        "x": ["listen_score_z","timecost_z","sponsor","listen_older","listen_female"],
        "names": ["ListenScore","TimeCost","Sponsor","ListenScore×Older","ListenScore×Female"],
        "cf_zero": ["Sponsor"]
    },
    "M2V7": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V8",
        "x": ["listen_score_z","timecost_z","sponsor","sponsor_lowIncome","sponsor_highIncome"],
        "names": ["ListenScore","TimeCost","Sponsor","Sponsor×LowIncome","Sponsor×HighIncome"],
        "cf_zero": ["Sponsor","Sponsor×LowIncome","Sponsor×HighIncome"]
    },
    "M2V8": {
        "table": f"{PROJECT_ID}.podcast_primary_bucket.panel_match_only_est_M2V10",
        "x": ["listen_score_z","timecost_z","sponsor","sponsor_female"],
        "names": ["ListenScore","TimeCost","Sponsor(Male ref)","Sponsor×Female"],
        "cf_zero": ["Sponsor(Male ref)","Sponsor×Female"]
    },
}

# ---- Pick which models to run (exclude V3_1 by design)
MODELS_TO_RUN = ["M1_baseline","M2V1","M2V2","M2V3","M2V4","M2V5","M2V6","M2V7","M2V8"]

def fetch_table(table_path:str, xcols:list):
    base_cols = ["user_id","podcast_id","chosen","listen_score_z","timecost_z","sponsor","obs_weight"]
    extra = [c for c in xcols if c not in base_cols]
    cols = list(dict.fromkeys(base_cols + extra))  # preserve order, dedupe
    q = f"SELECT {', '.join(cols)} FROM `{table_path}`"
    df = client.query(q).to_dataframe()
    # types & cleaning
    if "chosen" in df: df["chosen"] = pd.to_numeric(df["chosen"], errors="coerce").fillna(0).astype(int)
    if "sponsor" in df: df["sponsor"] = pd.to_numeric(df["sponsor"], errors="coerce").fillna(0).astype(int)
    if "obs_weight" not in df.columns or df["obs_weight"].isna().all():
        df["obs_weight"] = 1.0
    # numeric safety
    for c in [c for c in cols if c not in ["user_id","podcast_id","chosen"]]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    num_cols = [c for c in cols if c not in ["user_id","podcast_id","chosen"]]
    df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return df

def prepare_groups(df, xcols):
    # sort by user to form blocks
    order = np.argsort(df["user_id"].to_numpy())
    df = df.iloc[order].reset_index(drop=True)
    X = df[xcols].to_numpy(dtype=np.float64)
    y = df["chosen"].to_numpy(dtype=int)
    users = df["user_id"].to_numpy()
    w_row = df["obs_weight"].to_numpy(dtype=float)
    uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
    w_user = np.array([w_row[s:s+c].mean() for s,c in zip(start_idx,counts)], dtype=np.float64)
    return df, X, y, uniq_users, start_idx, counts, w_user

def fit_mnl(X, y, start_idx, counts, w_user, param_names):
    K = X.shape[1]
    def mnl_obj(beta):
        beta = np.asarray(beta, np.float64); ll = 0.0
        for i,(s,c) in enumerate(zip(start_idx,counts)):
            Xi = X[s:s+c]; yi = y[s:s+c]
            ui = Xi @ beta
            ll += w_user[i]*(ui[yi==1].sum() - logsumexp(ui))
        return -ll
    def mnl_score(beta):
        beta=np.asarray(beta,np.float64); g=np.zeros(K)
        for i,(s,c) in enumerate(zip(start_idx,counts)):
            Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta # yi is already sliced y[s:s+c]
            p=np.exp(ui-ui.max()); p/=p.sum()
            # Correct slicing for Xi[yi==1]
            g += w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T@p))
        return -g
    def mnl_hessian(beta):
        beta=np.asarray(beta,np.float64); H=np.zeros((K,K))
        for i,(s,c) in enumerate(zip(start_idx,counts)):
            Xi=X[s:s+c]; ui=Xi@beta
            p=np.exp(ui-ui.max()); p/=p.sum()
            Xw=Xi*p[:,None]; xbar=Xw.sum(axis=0)
            S=(Xi.T@(Xi*p[:,None]))-np.outer(xbar,xbar)
            H -= w_user[i]*S
        return H
    beta0 = np.zeros(K)
    opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
    beta_hat = opt.x
    H = mnl_hessian(beta_hat); H_inv = inv(H)
    # cluster-robust meat
    meat = np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta_hat
        p=np.exp(ui-ui.max()); p/=p.sum()
        score_i = w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T@p))
        meat += np.outer(score_i, score_i)
    cov = H_inv @ meat @ H_inv
    se  = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
    # fit stats
    LL  = -mnl_obj(beta_hat)
    LL0 = -np.sum(w_user * np.log(counts.astype(float)))
    r2  = 1 - (LL/LL0)
    r2a = 1 - ((LL - len(beta_hat))/LL0)
    res = pd.DataFrame({"coef":beta_hat, "std_err":se,
                        "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)},
                       index=param_names)
    return res, LL, LL0, r2, r2a, beta_hat

def cs_and_counterfactual(df, counts, uniq_users, param_names, beta_hat, cf_zero:list, xcols:list):
    # Build V baseline
    b = dict(zip(param_names, beta_hat))
    V = np.zeros(len(df))
    for name, col in zip(param_names, xcols):
        V += b[name] * df[col].to_numpy()
    # Per-user CS
    cs, ptr = [], 0
    for c in counts:
        Vi = V[ptr:ptr+c]
        cs.append(logsumexp(Vi))
        ptr += c
    cs = pd.Series(cs, index=uniq_users, name="CS")
    # Counterfactual Sponsor=0 (zero listed terms)
    V_cf = V.copy()
    # subtract the contribution of zeroed terms
    for name in cf_zero:
        if name in b:
            col = xcols[param_names.index(name)]
            V_cf -= b[name] * df[col].to_numpy()
    cs_cf, ptr = [], 0
    for c in counts:
        Vi = V_cf[ptr:ptr+c]
        cs_cf.append(logsumexp(Vi))
        ptr += c
    cs_cf = pd.Series(cs_cf, index=uniq_users, name="CS_noSponsor")
    delta = cs_cf - cs
    return cs, cs_cf, delta

# --- Helper function to format coefficients with values
BASE_TERMS = {"ListenScore","TimeCost","Sponsor","Sponsor(Male ref)"}

def format_coefficients(res_df, param_names, is_interaction=False):
    formatted_list = []
    for pname in param_names:
        is_base = pname in BASE_TERMS
        if (is_interaction and not is_base) or (not is_interaction and is_base):
            if pname in res_df.index:
                coef_value = res_df.loc[pname, "coef"]
                formatted_list.append(f"{pname} ({coef_value:.4f})")
    return ", ".join(formatted_list)

client = bigquery.Client(project=PROJECT_ID)

all_summaries = []

for model_key in MODELS_TO_RUN:
    cfg = REGISTRY[model_key]
    table = cfg["table"]; xcols = cfg["x"]; names = cfg["names"]; cf_zero = cfg["cf_zero"]

    print(f"\n\n================  {model_key}  ================")
    print(f"Table: `{table}`")
    print(f"Design: {names}")

    df = fetch_table(table, xcols)
    df, X, y, uniq_users, start_idx, counts, w_user = prepare_groups(df, xcols)

    res, LL, LL0, r2, r2a, beta_hat = fit_mnl(X, y, start_idx, counts, w_user, names)
    display(res.round(4))
    print(f"LL: {LL:.3f} | LL0(w): {LL0:.3f} | McFadden R^2: {r2:.4f} | Adj: {r2a:.4f}")

    cs, cs_cf, delta = cs_and_counterfactual(df, counts, uniq_users, names, beta_hat, cf_zero, xcols)
    print("\n--- Consumer Surplus (per user) ---")
    print(cs.describe().to_string())
    print("\n--- ΔCS under Sponsor=0 ---")
    print(delta.describe().to_string())

    # stash quick summary row
    row = {
        "model": model_key, "K": len(names),
        "LL": LL, "LL0_w": LL0, "McFadden_R2": r2, "Adj_R2": r2a,
        "N_users": len(counts), "N_rows": len(df),
        # Add new columns for formatted coefficients
        "Base Coefs (Value)": format_coefficients(res, names, is_interaction=False),
        "Interaction Coefs (Value)": format_coefficients(res, names, is_interaction=True)
    }
    all_summaries.append(row)

summary = pd.DataFrame(all_summaries) # No need to reindex if columns are consistent
print("\n\n============= Model Fit Summary =============")
display(summary.round(4))
